This shows that around the peak of the probability map we have a high density of information, hence we require more resolution

In [ ]:
ra, dec = ah.healpix_to_lonlat(ipix, nside, order='nested')
ra.deg, dec.deg

# Probability Density at a Known Position

In [ ]:
ra = 180 * u.deg
dec = 55 * u.deg

In [ ]:
max_level = 29
max_nside = ah.level_to_nside(max_level)
level, ipix = ah.uniq_to_level_ipix(skymap['UNIQ'])
index = ipix * (2**(max_level - level))**2

sorter = np.argsort(index)
match_ipix = ah.lonlat_to_healpix(ra, dec, max_nside, order='nested')
i = sorter[np.searchsorted(index, match_ipix, side='right', sorter=sorter)]
print('This is the probability density per deg^2:', skymap[i]['PROBDENSITY'].to_value(u.deg**-2))

Notice that now since each pixel has different area, we can only report probability per unit area

# Credible level at a Known Position

In [ ]:

import ligo.skymap.io.fits
from ligo.skymap.postprocess.util import find_greedy_credible_levels
from astropy.coordinates import SkyCoord

skymap_arr, _ = ligo.skymap.io.fits.read_sky_map('bayestar.multiorder.fits,1', nest=True)
cls = 100 * ligo.skymap.postprocess.util.find_greedy_credible_levels(skymap_arr)


theta = 0.5 * np.pi - np.deg2rad(-13.4780549)
phi = np.deg2rad(346.81640625)

# Query the multi-order map (handles variable resolution automatically)
ipix = hp.ang2pix(hp.get_nside(skymap_arr), theta, phi, nest=True)

print(f"Credible level : {cls[ipix]:.2f}%")

# 90% region

In [ ]:
#compute 90% area credible region from cls

#compute 90% area credible region from cls
a_90 = cls[cls <= 90]
n_pixels_90 = len(a_90)
area_90 = n_pixels_90 * hp.nside2pixarea(hp.get_nside(skymap_arr), degrees=True)
print(f"Number of pixels in 90% credible region: {n_pixels_90:,}")
print(f"Area of 90% credible region: {area_90:.2f} deg²")

# Probability inside a circle

In [ ]:
ra_test = 180
dec_test = 55
theta = np.radians(90 - dec_test)
phi = np.radians(ra_test)
radius = np.radians(5)  # example radius of 5 degrees
xyz = hp.ang2vec(theta, phi)
ipix_disc = hp.query_disc(hp.get_nside(skymap_arr), xyz, radius)
print(f"{m[ipix_disc].sum()*100:.2f}%")

# Let's plot the skymap with credible contours

In [ ]:
import ligo.skymap.io.fits
import ligo.skymap.postprocess.util
import numpy as np
import matplotlib.pyplot as pp
from matplotlib.colors import LogNorm
from matplotlib import cm
import ligo.skymap.plot

skymap, _ = ligo.skymap.io.fits.read_sky_map('bayestar.fits', nest = True) # NEST = True ensures that the order is rearranged properly, both for flat and multi-order maps

fig = pp.figure(dpi=300)
fig.patch.set_visible(False)
ax = pp.axes(projection='astro degrees mollweide')
ax.set_facecolor('none')
ax.grid()

cls = 100 * ligo.skymap.postprocess.util.find_greedy_credible_levels(skymap)

skymap[cls > 90] = np.nan
skymap[skymap == 0] = np.nan

ax.imshow_hpx((skymap, 'ICRS'), nested=True, cmap=cm.Oranges)
ax.contour_hpx((cls, 'ICRS'), nested=True, colors='black', levels=(50, 90), zorder=1, linestyles=['dashed', 'solid'], linewidths=0.5)

pp.show()


# Zoom

In [ ]:
# Use the peak location from earlier
fig = plt.figure()
fig.patch.set_visible(False)
ax = plt.axes(projection='astro zoom',
              center=f'{ra_deg:.4f}d {dec_deg:.4f}d', radius='30 deg')
ax.set_facecolor('none')
ax.grid()

ax.imshow_hpx((skymap, 'ICRS'), nested=True, cmap='Blues')

plt.title(f'Zoomed view around peak (RA={ra_deg:.2f}°, Dec={dec_deg:.2f}°)')
plt.show()


# Going in distance